In [4]:
# HW14 – эмбеддинги, FAISS, оценка retrieval и mini-RAG по базе знаний

# =========================
# 1. Импорты, seed и среда
# =========================

import os
import re
import random
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("artifacts", exist_ok=True)


Device: cpu


In [5]:
# ==========================================
# 2. База знаний и первичный sanity-check
# ==========================================

# Небольшая учебная база знаний по NumPy, pandas и Matplotlib

documents = [
    {
        "id": "numpy_intro",
        "text": "NumPy is a fundamental package for numerical computing in Python. It provides n-dimensional arrays and vectorized operations."
    },
    {
        "id": "numpy_indexing",
        "text": "NumPy arrays support slicing, boolean indexing and broadcasting. Indexing rules differ from Python lists."
    },
    {
        "id": "numpy_broadcasting",
        "text": "Broadcasting in NumPy allows operations on arrays of different shapes by expanding dimensions when possible."
    },
    {
        "id": "pandas_intro",
        "text": "Pandas is a library for data analysis. It provides Series and DataFrame structures for working with tabular data."
    },
    {
        "id": "pandas_groupby",
        "text": "The groupby operation in pandas allows splitting data, applying functions and combining results for aggregation."
    },
    {
        "id": "pandas_missing",
        "text": "Pandas has built-in support for missing values, including functions like isna, fillna and dropna."
    },
    {
        "id": "pandas_merge",
        "text": "Pandas merge and join operations combine DataFrames based on common columns or indices."
    },
    {
        "id": "matplotlib_intro",
        "text": "Matplotlib is a plotting library for creating static, animated and interactive visualizations in Python."
    },
    {
        "id": "matplotlib_plot",
        "text": "The basic Matplotlib workflow includes creating a figure, adding axes and calling plot or scatter functions."
    },
    {
        "id": "matplotlib_styles",
        "text": "Matplotlib supports styles and themes to control colors, fonts and overall appearance of plots."
    },
]

print("Number of documents:", len(documents))
pd.DataFrame(documents).head()


Number of documents: 10


,id,text
0,numpy_intro,NumPy is a fundamental package for numerical c...
1,numpy_indexing,"NumPy arrays support slicing, boolean indexing..."
2,numpy_broadcasting,Broadcasting in NumPy allows operations on arr...
3,pandas_intro,Pandas is a library for data analysis. It prov...
4,pandas_groupby,The groupby operation in pandas allows splitti...


In [6]:
# ==========================
# 3. Чанкинг документов
# ==========================

def simple_sentence_split(text: str):
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p]

chunks = []
chunk_id = 0

for doc in documents:
    sents = simple_sentence_split(doc["text"])
    for s in sents:
        chunks.append({
            "chunk_id": f"chunk_{chunk_id}",
            "doc_id": doc["id"],
            "text": s
        })
        chunk_id += 1

print("Number of chunks:", len(chunks))
pd.DataFrame(chunks).head()


Number of chunks: 13


,chunk_id,doc_id,text
0,chunk_0,numpy_intro,NumPy is a fundamental package for numerical c...
1,chunk_1,numpy_intro,It provides n-dimensional arrays and vectorize...
2,chunk_2,numpy_indexing,"NumPy arrays support slicing, boolean indexing..."
3,chunk_3,numpy_indexing,Indexing rules differ from Python lists.
4,chunk_4,numpy_broadcasting,Broadcasting in NumPy allows operations on arr...


In [7]:
# ==========================================
# 4. Эмбеддинги и индекс FAISS (baseline)
# ==========================================

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)

chunk_texts = [c["text"] for c in chunks]
embeddings = model.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)

# Нормируем для косинусного сходства через inner product
emb_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

d = emb_norm.shape[1]
index = faiss.IndexFlatIP(d)
index.add(emb_norm)

print("Index size:", index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index size: 13


In [8]:
# ==========================
# 5. Функция retrieval
# ==========================

def retrieve(query: str, top_k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True, show_progress_bar=False)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({
            "score": float(score),
            "chunk_id": chunks[idx]["chunk_id"],
            "doc_id": chunks[idx]["doc_id"],
            "text": chunks[idx]["text"]
        })
    return results

# Пример
retrieve("How to group data in pandas?", top_k=3)


[{'score': 0.724375307559967,
  'chunk_id': 'chunk_7',
  'doc_id': 'pandas_groupby',
  'text': 'The groupby operation in pandas allows splitting data, applying functions and combining results for aggregation.'},
 {'score': 0.5305968523025513,
  'chunk_id': 'chunk_9',
  'doc_id': 'pandas_merge',
  'text': 'Pandas merge and join operations combine DataFrames based on common columns or indices.'},
 {'score': 0.4886956214904785,
  'chunk_id': 'chunk_5',
  'doc_id': 'pandas_intro',
  'text': 'Pandas is a library for data analysis.'}]

In [9]:
# ==========================================
# 6. Контрольные запросы и оценка retrieval
# ==========================================

control_queries = [
    {"query": "What is NumPy used for?", "expected_doc": "numpy_intro"},
    {"query": "How does broadcasting work in arrays?", "expected_doc": "numpy_broadcasting"},
    {"query": "How to group data in pandas?", "expected_doc": "pandas_groupby"},
    {"query": "How to handle missing values in pandas?", "expected_doc": "pandas_missing"},
    {"query": "What is a DataFrame?", "expected_doc": "pandas_intro"},
    {"query": "How to merge two tables in pandas?", "expected_doc": "pandas_merge"},
    {"query": "How to create plots in Python?", "expected_doc": "matplotlib_intro"},
    {"query": "How to change plot style?", "expected_doc": "matplotlib_styles"},
]

TOP_K_BASE = 5

rows = []
for cq in control_queries:
    res = retrieve(cq["query"], top_k=TOP_K_BASE)
    retrieved_docs = [r["doc_id"] for r in res]
    hit = int(cq["expected_doc"] in retrieved_docs)
    rank = retrieved_docs.index(cq["expected_doc"]) + 1 if cq["expected_doc"] in retrieved_docs else None
    rows.append({
        "query": cq["query"],
        "expected_source": cq["expected_doc"],
        "retrieved_sources": ",".join(retrieved_docs),
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

retrieval_df = pd.DataFrame(rows)
retrieval_df


,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,What is NumPy used for?,numpy_intro,"numpy_intro,numpy_indexing,numpy_broadcasting,...",1,1
1,How does broadcasting work in arrays?,numpy_broadcasting,"numpy_broadcasting,numpy_indexing,numpy_intro,...",1,1
2,How to group data in pandas?,pandas_groupby,"pandas_groupby,pandas_merge,pandas_intro,panda...",1,1
3,How to handle missing values in pandas?,pandas_missing,"pandas_missing,pandas_merge,pandas_intro,panda...",1,1
4,What is a DataFrame?,pandas_intro,"pandas_intro,pandas_intro,pandas_groupby,panda...",1,1
5,How to merge two tables in pandas?,pandas_merge,"pandas_merge,pandas_groupby,pandas_intro,panda...",1,1
6,How to create plots in Python?,matplotlib_intro,"matplotlib_plot,matplotlib_intro,matplotlib_st...",1,2
7,How to change plot style?,matplotlib_styles,"matplotlib_styles,matplotlib_plot,matplotlib_i...",1,1


In [10]:
# ==========================
# 7. Метрики hit@k, recall@k, MRR@k
# ==========================

hit_at_k = retrieval_df["hit_at_k"].mean()
# Здесь по одному релевантному документу на запрос, поэтому recall@k = hit@k
recall_at_k = hit_at_k
mrr = retrieval_df["rank_of_first_relevant"].apply(lambda r: 1/r if r is not None else 0).mean()

print("hit@{}: {:.3f}".format(TOP_K_BASE, hit_at_k))
print("recall@{}: {:.3f}".format(TOP_K_BASE, recall_at_k))
print("MRR@{}: {:.3f}".format(TOP_K_BASE, mrr))

retrieval_df.to_csv("artifacts/retrieval_eval.csv", index=False)


hit@5: 1.000
recall@5: 1.000
MRR@5: 0.938


In [11]:
# ==========================================
# 8. Эксперимент с параметром top_k (3 vs 5)
# ==========================================

def eval_with_top_k(top_k: int):
    hits = []
    for cq in control_queries:
        res = retrieve(cq["query"], top_k=top_k)
        retrieved_docs = [r["doc_id"] for r in res]
        hit = int(cq["expected_doc"] in retrieved_docs)
        hits.append(hit)
    return np.mean(hits)

hit_k3 = eval_with_top_k(3)
hit_k5 = eval_with_top_k(5)

print("hit@3:", hit_k3)
print("hit@5:", hit_k5)


hit@3: 1.0
hit@5: 1.0


In [12]:
# ==========================================
# 9. Обновление базы знаний и переиндексация
# ==========================================

new_documents = [
    {
        "id": "matplotlib_subplots",
        "text": "Matplotlib supports subplots to arrange multiple plots in a single figure."
    },
    {
        "id": "matplotlib_legends",
        "text": "Legends in Matplotlib help label different elements of a plot for better readability."
    },
    {
        "id": "numpy_performance",
        "text": "NumPy operations are implemented in C and can be much faster than pure Python loops."
    },
]

documents_updated = documents + new_documents

chunks_updated = []
chunk_id = 0
for doc in documents_updated:
    sents = simple_sentence_split(doc["text"])
    for s in sents:
        chunks_updated.append({
            "chunk_id": f"chunk_{chunk_id}",
            "doc_id": doc["id"],
            "text": s
        })
        chunk_id += 1

print("Number of chunks after update:", len(chunks_updated))

chunk_texts_upd = [c["text"] for c in chunks_updated]
emb_upd = model.encode(chunk_texts_upd, convert_to_numpy=True, show_progress_bar=False)
emb_upd = emb_upd / np.linalg.norm(emb_upd, axis=1, keepdims=True)

index_upd = faiss.IndexFlatIP(d)
index_upd.add(emb_upd)

def retrieve_updated(query: str, top_k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True, show_progress_bar=False)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores, idxs = index_upd.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        results.append({
            "score": float(score),
            "chunk_id": chunks_updated[idx]["chunk_id"],
            "doc_id": chunks_updated[idx]["doc_id"],
            "text": chunks_updated[idx]["text"]
        })
    return results


Number of chunks after update: 16


In [13]:
# ==========================================
# 10. Сравнение retrieval до/после обновления
# ==========================================

rows_before_after = []
for cq in control_queries:
    before = retrieve(cq["query"], top_k=TOP_K_BASE)
    after = retrieve_updated(cq["query"], top_k=TOP_K_BASE)
    before_docs = [r["doc_id"] for r in before]
    after_docs = [r["doc_id"] for r in after]
    changed = int(before_docs != after_docs)
    rows_before_after.append({
        "query": cq["query"],
        "before_retrieved_sources": ",".join(before_docs),
        "after_retrieved_sources": ",".join(after_docs),
        "changed": changed
    })

before_after_df = pd.DataFrame(rows_before_after)
before_after_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
before_after_df


,query,before_retrieved_sources,after_retrieved_sources,changed
0,What is NumPy used for?,"numpy_intro,numpy_indexing,numpy_broadcasting,...","numpy_intro,numpy_performance,numpy_indexing,n...",1
1,How does broadcasting work in arrays?,"numpy_broadcasting,numpy_indexing,numpy_intro,...","numpy_broadcasting,numpy_indexing,numpy_intro,...",1
2,How to group data in pandas?,"pandas_groupby,pandas_merge,pandas_intro,panda...","pandas_groupby,pandas_merge,pandas_intro,panda...",0
3,How to handle missing values in pandas?,"pandas_missing,pandas_merge,pandas_intro,panda...","pandas_missing,pandas_merge,pandas_intro,panda...",0
4,What is a DataFrame?,"pandas_intro,pandas_intro,pandas_groupby,panda...","pandas_intro,pandas_intro,pandas_groupby,panda...",0
5,How to merge two tables in pandas?,"pandas_merge,pandas_groupby,pandas_intro,panda...","pandas_merge,pandas_groupby,pandas_intro,panda...",0
6,How to create plots in Python?,"matplotlib_plot,matplotlib_intro,matplotlib_st...","matplotlib_plot,matplotlib_intro,matplotlib_su...",1
7,How to change plot style?,"matplotlib_styles,matplotlib_plot,matplotlib_i...","matplotlib_styles,matplotlib_legends,matplotli...",1


In [14]:
# ==========================================
# 11. Mini-RAG: контекст + ответ + источники
# ==========================================

def build_context(results, max_chunks: int = 3):
    return " ".join([r["text"] for r in results[:max_chunks]])

def mini_rag_answer(question: str, top_k: int = 5):
    res = retrieve_updated(question, top_k=top_k)
    context = build_context(res, max_chunks=3)
    # Простой учебный генератор ответа
    answer = f"Based on the documentation: {context}"
    sources = list({r["doc_id"] for r in res})
    return answer, sources, res

test_questions = [
    "How can I group data in pandas?",
    "How to handle missing values in pandas?",
    "How to create multiple plots in one figure?",
    "What is broadcasting in NumPy?",
    "How to change plot style in Matplotlib?"
]

rag_rows = []
for q in test_questions:
    ans, src, res = mini_rag_answer(q, top_k=TOP_K_BASE)
    rag_rows.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": ",".join(src)
    })

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
rag_df


,question,answer,retrieved_sources
0,How can I group data in pandas?,Based on the documentation: The groupby operat...,"pandas_intro,pandas_merge,pandas_missing,panda..."
1,How to handle missing values in pandas?,Based on the documentation: Pandas has built-i...,"pandas_intro,pandas_merge,numpy_intro,pandas_m..."
2,How to create multiple plots in one figure?,Based on the documentation: Matplotlib support...,"matplotlib_subplots,matplotlib_plot,matplotlib..."
3,What is broadcasting in NumPy?,Based on the documentation: Broadcasting in Nu...,"numpy_indexing,numpy_intro,numpy_broadcasting,..."
4,How to change plot style in Matplotlib?,Based on the documentation: Matplotlib support...,"matplotlib_subplots,matplotlib_plot,matplotlib..."
